# Notebook 05: SATA Architecture and Training

**Purpose**: Define, train, and validate SATA on synthetic tasks.

**Gate 2 (end of Week 5)**: does SATA top-k selection beat the best protocol and random selection on validation tasks (XGBoost proxy accuracy)? If not, RQ4 becomes a rigorous negative result — document why and pivot to ablation analysis.

## What SATA is — and, explicitly, what it is not

This is worth stating plainly because it was genuinely unclear during the literature-review stage (see the margin questions in `Lit-review.pdf` around §2.4.1): **SATA never predicts a label.** It has no classification head, is never shown a label to predict, and its output is not a prediction — it's a set of relevance *scores* over the demo pool. The frozen base LLM (Llama-3.1-8B / Qwen2.5-7B) is the only thing in this whole pipeline that ever classifies anything, via ordinary text-serialised ICL (Notebook 01/02). SATA's entire job is choosing *which* rows from the pool get serialised into that LLM's prompt, and in what order of relevance.

This makes SATA architecturally unlike **TabPFN** (Hollmann et al.) — TabPFN *is* the classifier, trained end-to-end to map a support set + query directly to a label in one forward pass. It's also unlike simply "training a pretrained LLM further" — that would still leave the model doing its own classification, just with different weights, and would break the "frozen base model" premise the whole faithfulness evaluation (Notebook 03) depends on. SATA is a small, separately-trained **demonstration selector** that sits *upstream* of an unmodified frozen LLM.

## Where the architecture comes from

SATA's design is adapted from **In-Context Risk Minimization / "Context is environment"** (Gupta, Jegelka, Lopez-Paz & Ahuja, 2023, Lit-review §3 ref [31]) — the finding that a transformer attending over contextual examples *at inference time* can learn to extract environment-specific representations that improve OOD generalisation, without touching the base predictor's weights. SATA specialises this idea to the tabular ICL setting: instead of learning environment representations for a downstream classifier it owns, SATA learns to score *which demonstrations* would make the environment/regime legible to a downstream classifier (here, the frozen LLM) that it doesn't own or modify.

Concretely, per `src/models/sata.py`: demo features + demo labels get embedded together (`feature_embed(demo_features) + label_embed(demo_labels)`), the query gets embedded from its features alone (it has no label yet — that's what's being predicted), all tokens go through a small transformer encoder with self-attention (so every demo's relevance can depend on every other demo *and* on the query), and a scoring head converts each demo position's output into a single relevance logit. A softmax over demo positions turns those logits into a probability distribution — the top-k highest-probability demos are what get serialised into the LLM's prompt in Notebook 06.

**Why query-conditioned, specifically?** This is SATA's core empirical bet, and it's what the query-agnostic ablation (below) exists to test in isolation. Goddard et al.'s (2025) task-diversity result (Lit-review §2.3.1) shows ICL-like systems undergo a sharp phase transition from "solutions specialised to pretraining-like tasks" to "solutions that generalise across the task space" once pretraining task diversity crosses a threshold. That's a lever at *training* time. SATA's hypothesis is that an analogous lever exists at *inference* time: if which demonstrations are relevant genuinely depends on the specific query (not just the task in the abstract), a selector that conditions its scores on the query should transfer better across environments than one that scores the pool once and reuses the same ranking for every query in a task.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Architecture

See `src/models/sata.py::SATA` and `SATAQueryAgnostic` (ablation: masks the query token so scores don't depend on query identity).

`SATA.forward(demo_features, demo_labels, query_features) -> scores` where `scores` has shape `(batch, n_demos)` and sums to 1 (softmax over demo positions). `SATAQueryAgnostic` overrides `forward` to zero out the query before the parent's logic runs — everything else (embeddings, transformer, scoring head) is identical, so any accuracy difference between the two models isolates the effect of query-conditioning specifically, not some other architectural change. See the intro above for why that isolation is the point of this ablation.

In [2]:
from src.models.sata import SATA, SATAQueryAgnostic

model = SATA(
    n_features=config.generator.n_features,
    d_model=config.sata.d_model,
    n_heads=config.sata.n_heads,
    n_layers=config.sata.n_layers,
)
model

SATA(
  (feature_embed): Linear(in_features=10, out_features=128, bias=True)
  (label_embed): Embedding(2, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (score_head): Linear(in_features=128, out_features=1, bias=True)
)

## Target score computation

See `src/models/sata_targets.py::compute_target_scores` — high weight for same-regime and counter-spurious demos, mild bonus for label match, near-zero for spurious-only demos from irrelevant regimes.

**This is SATA's supervision signal, and it's deliberately not "does the demo have the same label as the query."** A demo that merely shares the query's label but comes from an unrelated regime, or agrees with the query only via the spurious feature, is exactly the kind of demonstration that would teach the frozen LLM a shortcut rather than the task's real structure — the same failure mode Section 2.2 of the lit review documents at the model level. Weighting by **same regime** (does this demo sit in the same decision-rule leaf/branch as the query?) rewards structural relevance: a demo from the query's own region of the input space is informative about the local decision boundary regardless of whether its label happens to match. Weighting by **counter-spurious** (does this demo's spurious-feature value disagree with its label?) is the training-time analogue of Notebook 02's counter-spurious diversity protocol — it directly rewards demonstrations that *can't* be explained by the shortcut. The label-match term is kept deliberately mild (`+0.5`, vs. `+2.0` for regime and `+1.5` for counter-spurious) so SATA doesn't degenerate into a label-matching heuristic, which per Min et al. (2022, Lit-review §2.6.1) isn't even what drives ICL performance in the first place.

## Training loop

See `src/models/sata_train.py::train_sata` (KL-divergence loss) and `evaluate_sata_proxy` (XGBoost proxy validation, no LLM calls).

**Why KL divergence, not cross-entropy against a single "correct" demo?** SATA's target (above) is a full probability distribution over the demo pool, not a one-hot label — several demos can be simultaneously relevant. KL divergence trains SATA's predicted distribution to match that target distribution's *shape*, not just to spike on one "best" demo, which is the right objective when the supervision itself is graded relevance rather than a single ground-truth choice.

**Why validate with an XGBoost proxy instead of the frozen LLM?** Running the actual frozen LLM once per validation task per epoch would be prohibitively slow and expensive — validation needs to run every epoch, for potentially 50 epochs, just to pick the best checkpoint. `evaluate_sata_proxy` substitutes a cheap, deterministic stand-in: fit an XGBoost classifier on SATA's top-k selected demos, then check whether it predicts the query correctly. This isn't a claim that XGBoost behaves like the LLM — it's a fast, LLM-free signal for "did SATA select demos that are informative about the query's regime," which is exactly the property SATA's training target was built to reward. The real test of whether this transfers to the frozen LLM happens downstream in Notebook 06, which does use the actual LLM.

**Early stopping and crash resume.** The loop originally always ran the full `sata.epochs` (50) regardless of how val_proxy behaved — on the first real run, val_proxy peaked at epoch 0 and never recovered through epoch 35+, meaning most of a ~5-hour run was spent on epochs that never improved the checkpoint actually used downstream. `train_sata` now stops once `sata.patience` (10) epochs pass with no val_proxy improvement over the best seen so far; 10 is deliberately mild given val_proxy is a somewhat noisy proxy metric (a point or two of movement between consecutive epochs on its own), so a short patience risks stopping on noise rather than a genuine plateau. Separately, a multi-hour run that gets interrupted (kernel killed, walltime hit, OnDemand connection dropped — this happened mid-run once already) previously had no way to resume; it restarted from epoch 0 every time. `train_sata` now also takes a `resume_checkpoint_path`: full training state (model + optimizer, current epoch, best-so-far bookkeeping, log-so-far) is written there after every epoch, and automatically loaded from if the file already exists, so an interrupted run picks up at the next epoch instead of redoing everything. That file is deleted automatically once training actually finishes (full epoch budget or early stop) so a later *intentional* fresh run (e.g. after changing hyperparameters) doesn't silently inherit stale state — delete it manually only if you need to abandon an in-progress run and start over.

In [3]:
import json
from types import SimpleNamespace

import numpy as np
import pandas as pd

from src.data.generator import SyntheticTask
from src.models.sata_train import train_sata, evaluate_sata_proxy

# train_sata/evaluate_sata_proxy read config.lr/.epochs/.max_demos/... (the `sata`
# block) *and* config.environments (the `generator` block) -- merge them so a
# single namespace satisfies both.
sata_cfg = SimpleNamespace(**vars(config.sata), environments=config.generator.environments)

SYN_ROOT = resolve_path(config.paths.data_synthetic)
gen_meta = json.load(open(SYN_ROOT / 'generator_config.json'))
if not gen_meta.get('frozen', False):
    print(f"Warning: generator gate not passed (pass_rate={gen_meta['gate_pass_rate']:.0%}) — "
          "training against Notebook 04's provisional data. Re-run Notebook 04 once it's frozen "
          "before treating anything downstream of this as a final result.")


def load_task_suite(group_dir):
    tasks = []
    for meta_path in sorted(Path(group_dir).glob('*_meta.json')):
        meta = json.load(open(meta_path))
        tasks.append(SyntheticTask(
            task_id=meta['task_id'],
            rule_family=meta['rule_family'],
            causal_features=meta['causal_features'],
            coefficients=np.array(meta['coefficients']),
            spurious_strength=meta['spurious_strength'],
            n_features=config.generator.n_features,
            label_noise=config.generator.label_noise,
            threshold=meta['threshold'],
            # 'threshold'/'tree' families only (see src/data/generator.py);
            # None for 'linear'/'sparse_interaction', which is what
            # SyntheticTask's defaults already are.
            thresholds3=meta.get('thresholds3'),
            leaf_labels=meta.get('leaf_labels'),
        ))
    return tasks


train_tasks = load_task_suite(SYN_ROOT / 'tasks_train')
val_tasks = load_task_suite(SYN_ROOT / 'tasks_val')
print(f"Loaded {len(train_tasks)} train tasks, {len(val_tasks)} val tasks")

resolve_path('models').mkdir(parents=True, exist_ok=True)
# resume_checkpoint_path: if this file already exists (e.g. the kernel died
# or lost its connection partway through a multi-hour run), train_sata picks
# back up at the next epoch instead of restarting from epoch 0 -- delete the
# file first if you deliberately want a fresh run. Separate from
# checkpoint_path, which stays a bare state_dict for downstream loading.
log = train_sata(
    model, train_tasks, val_tasks, sata_cfg,
    checkpoint_path=resolve_path('models/sata_best.pt'),
    resume_checkpoint_path=resolve_path('models/sata_best_resume.pt'),
)
training_log_df = pd.DataFrame(log)
training_log_df['model'] = 'sata'
training_log_df

Loaded 2000 train tasks, 200 val tasks
Resumed from /home/562/cg3543/LLM-ICL-OOD-Honours/sata-project/models/sata_best_resume.pt: starting at epoch 3, best_val_score=0.9587 (epoch 0)


Epoch 3: loss=0.2446, val_proxy=0.9363


Epoch 4: loss=0.2328, val_proxy=0.9311


Epoch 5: loss=0.2233, val_proxy=0.9353


Epoch 6: loss=0.2140, val_proxy=0.9448


Epoch 7: loss=0.2037, val_proxy=0.9380


Epoch 8: loss=0.1944, val_proxy=0.9219


Epoch 9: loss=0.1882, val_proxy=0.9287


Epoch 10: loss=0.1842, val_proxy=0.9333
Early stopping at epoch 10 (no val_proxy improvement in 10 epochs) -- best val_proxy=0.9587 at epoch 0.


,epoch,loss,val_proxy,model
0,0,0.296824,0.958750,sata
1,1,0.265844,0.951719,sata
2,2,0.251416,0.953438,sata
3,3,0.244597,0.936250,sata
4,4,0.232767,0.931094,sata
5,5,0.223318,0.935312,sata
6,6,0.214009,0.944844,sata
7,7,0.203732,0.937969,sata
8,8,0.194412,0.921875,sata
9,9,0.188247,0.928750,sata


## SATA ablation: query-agnostic variant

Trained identically to the main model (same data, same loss, same schedule) — the only difference is `SATAQueryAgnostic` zeroing the query before scoring (see the Architecture cell above). If this variant scores nearly as well as full SATA in the Gate 2 check below, query-conditioning isn't pulling its weight and RQ4's story becomes "demonstration reweighting helps, but not because it's query-specific."

In [2]:
query_agnostic_model = SATAQueryAgnostic(
    n_features=config.generator.n_features,
    d_model=config.sata.d_model,
    n_heads=config.sata.n_heads,
    n_layers=config.sata.n_layers,
)
query_agnostic_log = train_sata(
    query_agnostic_model, train_tasks, val_tasks, sata_cfg,
    checkpoint_path=resolve_path('models/sata_query_agnostic.pt'),
    resume_checkpoint_path=resolve_path('models/sata_query_agnostic_resume.pt'),
)
query_agnostic_log_df = pd.DataFrame(query_agnostic_log)
query_agnostic_log_df['model'] = 'sata_query_agnostic'
query_agnostic_log_df

NameError: name 'SATAQueryAgnostic' is not defined

## Gate 2 check

Compare SATA top-k vs. best protocol vs. random on validation tasks via `evaluate_sata_proxy`.

**Why beating both baselines, not just one, is the bar.** Beating random alone would only show that *some* form of non-uniform selection helps — which Notebook 02 will already have established for the hand-designed protocols. Beating the best hand-designed protocol specifically is what would justify the claim that *learned, query-conditioned* reweighting adds value beyond what a human-designed heuristic already captures — that's RQ4's actual question. If Gate 2 fails here, the spec is explicit about what that means: RQ4 becomes a **rigorous negative result**, not a bug to fix by relaxing the criterion. A negative result — "hand-designed diversity protocols already capture what's available; a learned reweighter adds nothing measurable" — is still a real, publishable finding, and the honest thing to do is report it and pivot to ablation analysis (which ingredient of SATA, if any, helps) rather than keep tuning until the number moves.

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

import torch
from src.models.sata_train import _task_batch, _fit_predict_one

# Gate 2 evaluates the *best* checkpoint (highest val_proxy during training),
# not whatever epoch the loop happened to end on.
model.load_state_dict(torch.load(resolve_path('models/sata_best.pt'), weights_only=True))

# Ground-truth-tag-based protocol proxies, mirroring src/selection's real-arm
# protocols but operating directly on generate_environment's metadata dicts
# (regime/is_counter_spurious/label) rather than a pandas pool -- avoids a
# numpy<->DataFrame round trip just for this proxy comparison.

def select_random(demo_meta, k, seed):
    rng = np.random.default_rng(seed)
    return rng.choice(len(demo_meta), size=k, replace=False)


def select_label_diversity(demo_meta, k, seed):
    rng = np.random.default_rng(seed)
    labels = np.array([m['label'] for m in demo_meta])
    classes = np.unique(labels)
    per_class = k // len(classes)
    selected = []
    for c in classes:
        idx = np.where(labels == c)[0]
        selected.extend(rng.choice(idx, size=min(per_class, len(idx)), replace=False))
    remaining = k - len(selected)
    if remaining > 0:
        leftover = np.setdiff1d(np.arange(len(demo_meta)), selected)
        selected.extend(rng.choice(leftover, size=min(remaining, len(leftover)), replace=False))
    return np.array(selected[:k])


def select_rule_diversity(demo_meta, k, seed):
    rng = np.random.default_rng(seed)
    regimes = np.array([m['regime'] for m in demo_meta])
    groups = {}
    for i, r in enumerate(regimes):
        groups.setdefault(r, []).append(i)
    keys = list(groups.keys())
    rng.shuffle(keys)
    selected = []
    for r in keys:
        if len(selected) >= k:
            break
        selected.append(rng.choice(groups[r]))
    remaining = k - len(selected)
    if remaining > 0:
        leftover = np.setdiff1d(np.arange(len(demo_meta)), selected)
        selected.extend(rng.choice(leftover, size=min(remaining, len(leftover)), replace=False))
    return np.array(selected[:k])


def select_counter_spurious(demo_meta, k, seed):
    rng = np.random.default_rng(seed)
    counter_idx = np.array([i for i, m in enumerate(demo_meta) if m['is_counter_spurious']])
    n_counter = min(k // 2 + 1, len(counter_idx))
    selected = list(rng.choice(counter_idx, size=n_counter, replace=False)) if n_counter else []
    remaining = k - len(selected)
    if remaining > 0:
        leftover = np.setdiff1d(np.arange(len(demo_meta)), selected)
        selected.extend(rng.choice(leftover, size=min(remaining, len(leftover)), replace=False))
    return np.array(selected[:k])


PROTOCOLS = {
    'random': select_random,
    'label_diversity': select_label_diversity,
    'rule_diversity': select_rule_diversity,
    'counter_spurious': select_counter_spurious,
}


def evaluate_protocol_proxy(select_fn, val_tasks, sata_cfg, k, n_jobs=None):
    if n_jobs is None:
        try:
            n_jobs = len(os.sched_getaffinity(0))
        except AttributeError:
            n_jobs = os.cpu_count() or 4

    accs = []
    # Same rationale as src/models/sata_train.py::evaluate_sata_proxy: each
    # query's XGBoost fit is independent and tiny, and XGBoost's C++ fit
    # releases the GIL, so a thread pool turns this into real parallelism
    # across the job's allocated cores instead of one fit at a time.
    with ThreadPoolExecutor(max_workers=n_jobs) as pool:
        for task in val_tasks:
            (demo_X, demo_y, demo_meta), (query_X, query_y, query_meta) = _task_batch(
                task, 'id', n_demos=sata_cfg.max_demos, n_queries=32
            )
            idx_per_query = [select_fn(demo_meta, k, seed=i) for i in range(query_X.shape[0])]
            futures = [
                pool.submit(_fit_predict_one, demo_X[idx], demo_y[idx], query_X[i], query_y[i])
                for i, idx in enumerate(idx_per_query)
            ]
            accs.extend(f.result() for f in futures)
    return float(np.mean(accs)) if accs else 0.0


sata_val_score = evaluate_sata_proxy(model, val_tasks, sata_cfg, k=config.k_primary)
protocol_scores = {
    name: evaluate_protocol_proxy(fn, val_tasks, sata_cfg, k=config.k_primary) for name, fn in PROTOCOLS.items()
}
best_protocol_name = max(protocol_scores, key=protocol_scores.get)
best_protocol_score = protocol_scores[best_protocol_name]

gate2_passed = sata_val_score > best_protocol_score and sata_val_score > protocol_scores['random']

print(f"SATA proxy accuracy: {sata_val_score:.4f}")
for name, score in protocol_scores.items():
    print(f"  {name}: {score:.4f}")
print(f"Best protocol: {best_protocol_name} ({best_protocol_score:.4f})")
print(
    "GATE 2 PASSED — SATA beats both random and the best protocol." if gate2_passed
    else "GATE 2 FAILED — RQ4 becomes a rigorous negative result; document why and pivot to ablation analysis."
)

gate2_summary = pd.DataFrame(
    [{'method': 'sata', 'proxy_accuracy': sata_val_score}]
    + [{'method': name, 'proxy_accuracy': score} for name, score in protocol_scores.items()]
)
gate2_summary

## Output

- `models/sata_best.pt` — best checkpoint by validation loss
- `models/sata_query_agnostic.pt` — ablation checkpoint
- `models/sata_best_resume.pt` / `models/sata_query_agnostic_resume.pt` — full resume state, only present while a run is genuinely in progress or was interrupted; auto-deleted on normal completion (see "Training loop" above)
- `results/sata_training_log.parquet` — loss curves, validation metrics per epoch

In [ ]:
# Checkpoints (models/sata_best.pt, models/sata_query_agnostic.pt) are already
# saved incrementally by train_sata's checkpoint_path -- nothing to do here but
# persist the logs.
resolve_path('results').mkdir(parents=True, exist_ok=True)

full_log = pd.concat([training_log_df, query_agnostic_log_df], ignore_index=True)
full_log.to_parquet(resolve_path('results/sata_training_log.parquet'), index=False)
gate2_summary.to_parquet(resolve_path('results/sata_gate2_summary.parquet'), index=False)

print(f"Saved {len(full_log)}-row training log + Gate 2 summary. "
      f"Checkpoints: models/sata_best.pt, models/sata_query_agnostic.pt")